In [ ]:
!pip -q install earthengine-api geemap pandas numpy ipywidgets scikit-learn

import os, re, glob
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Safe Google Drive mount / path detection
# ------------------------------------------------------------
def get_drive_base():
    for b in ["/content/drive/MyDrive", "/content/gdrive/MyDrive"]:
        if os.path.exists(b):
            return b

    from google.colab import drive

    try:
        drive.mount("/content/drive")
        if os.path.exists("/content/drive/MyDrive"):
            return "/content/drive/MyDrive"
    except Exception as e:
        print("Mount at /content/drive failed:", e)

    drive.mount("/content/gdrive", force_remount=True)
    if os.path.exists("/content/gdrive/MyDrive"):
        return "/content/gdrive/MyDrive"

    raise FileNotFoundError("Could not mount Google Drive.")

BASE_DRIVE = get_drive_base()
print("Using BASE_DRIVE:", BASE_DRIVE)

PEST_DIR = os.path.join(BASE_DRIVE, "Pest")
GEE_INPUTS_DIR = os.path.join(PEST_DIR, "gee_inputs")
GEE_OUTPUTS_DIR = os.path.join(PEST_DIR, "gee_outputs")

os.makedirs(GEE_INPUTS_DIR, exist_ok=True)
os.makedirs(GEE_OUTPUTS_DIR, exist_ok=True)

RAW_CSV = os.path.join(PEST_DIR, "pests_2015_2025 new.csv")
CLEAN_UNIQUE_CSV = os.path.join(GEE_INPUTS_DIR, "rice_top10_pest_points_unique_for_gee.csv")

print("RAW_CSV exists:", os.path.exists(RAW_CSV))
print("CLEAN_UNIQUE_CSV exists:", os.path.exists(CLEAN_UNIQUE_CSV))

TOP10_RICE_GROUPS = [
    "Brown planthopper",
    "Stem borer",
    "Rodents",
    "Bacterial leaf blight",
    "Leaffolder",
    "Rice blast",
    "Rice black bug",
    "Rice bug complex",
    "Bacterial leaf streak",
    "Green leafhopper"
]

MONTH_MAP = {
    "jan": 1, "january": 1,
    "feb": 2, "february": 2,
    "mar": 3, "march": 3,
    "apr": 4, "april": 4,
    "may": 5,
    "jun": 6, "june": 6,
    "jul": 7, "july": 7,
    "aug": 8, "august": 8,
    "sep": 9, "sept": 9, "september": 9,
    "oct": 10, "october": 10,
    "nov": 11, "november": 11,
    "dec": 12, "december": 12
}

def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def parse_year(x):
    s = clean_text(x)
    m = re.search(r"\b(20[0-3][0-9]|19[8-9][0-9])\b", s)
    return int(m.group(1)) if m else np.nan

def parse_month(x):
    s = clean_text(x).lower()
    if s in MONTH_MAP:
        return MONTH_MAP[s]
    for k, v in MONTH_MAP.items():
        if k in s:
            return v
    m = re.search(r"\b(1[0-2]|0?[1-9])\b", s)
    return int(m.group(1)) if m else np.nan

def dms_to_decimal(deg, minute=0, sec=0, direction=None):
    val = float(deg) + float(minute) / 60.0 + float(sec) / 3600.0
    if direction and direction.upper() in ["S", "W"]:
        val = -val
    return val

def parse_coord(x, want="lat"):
    s = clean_text(x)
    if s == "":
        return np.nan

    s = (
        s.replace("º", "°")
         .replace("˚", "°")
         .replace("’", "'")
         .replace("‘", "'")
         .replace("”", '"')
         .replace("“", '"')
    )

    dirs = "NS" if want == "lat" else "EW"

    # N17°34'5"
    m = re.search(rf"([{dirs}])\s*(\d{{1,3}})[°\s]+(\d{{1,2}})[\'\s]+(\d{{1,3}}(?:\.\d+)?)", s, flags=re.I)
    if m:
        return dms_to_decimal(m.group(2), m.group(3), m.group(4), m.group(1))

    # 17°34'5"N
    m = re.search(rf"(\d{{1,3}})[°\s]+(\d{{1,2}})[\'\s]+(\d{{1,3}}(?:\.\d+)?)\s*([{dirs}])", s, flags=re.I)
    if m:
        return dms_to_decimal(m.group(1), m.group(2), m.group(3), m.group(4))

    # decimal
    nums = re.findall(r"-?\d{1,3}(?:\.\d+)?", s)
    if nums:
        val = float(nums[0])
        if want == "lat" and -90 <= val <= 90:
            return val
        if want == "lon" and -180 <= val <= 180:
            return val

    return np.nan

def standardize_rice_pest(x):
    s = clean_text(x).lower()
    s2 = re.sub(r"[^a-z0-9]+", " ", s).strip()

    rules = [
        (r"\bbrown\s*planthopper\b|\bbph\b", "Brown planthopper"),
        (r"\bstem\s*borer\b|\bstemborer\b|\byellow\s*stem\s*borer\b", "Stem borer"),
        (r"\brodent\b|\brodents\b|\brat\b|\brats\b|\bmouse\b|\bmice\b", "Rodents"),
        (r"\bbacterial\s*leaf\s*blight\b|\bblb\b|\bkresek\b", "Bacterial leaf blight"),
        (r"\bleaf\s*folder\b|\bleaffolder\b", "Leaffolder"),
        (r"\brice\s*blast\b|\bblast\b|\bneck\s*blast\b|\bcollar\s*blast\b|\bleaf\s*blast\b|\bnode\s*blast\b|\bpanicle\s*blast\b|\bsheath\s*blast\b", "Rice blast"),
        (r"\brice\s*black\s*bug\b|\bblack\s*bug\b", "Rice black bug"),
        (r"\brice\s*bug\b|\brice\s*grain\s*bug\b|\brice\s*stink\s*bug\b|\bstink\s*bug\b|\bwhite\s*spotted\s*stink\s*bug\b", "Rice bug complex"),
        (r"\bbacterial\s*leaf\s*streak\b|\bbls\b", "Bacterial leaf streak"),
        (r"\bgreen\s*leafhopper\b|\bglh\b|\bzigzag\s*leafhopper\b", "Green leafhopper"),
    ]

    for pat, label in rules:
        if re.search(pat, s2):
            return label

    return "Other"

def standardize_stage(x):
    s = clean_text(x).lower()
    s2 = re.sub(r"[^a-z0-9]+", " ", s).strip()

    if re.search(r"nursery|seedling|germin|emerg|sow|sown|re sown|re planted|transplant|establish", s2):
        return "establishment_seedling"
    if re.search(r"vegetative|tiller|tillering|maximum tiller|active tiller|elongation", s2):
        return "vegetative"
    if re.search(r"panicle initiation|boot|booting|heading|flower|flowering|anthesis|reproductive|milking|milk", s2):
        return "reproductive"
    if re.search(r"dough|ripen|ripening|matur|maturity|harvest", s2):
        return "maturity_harvest"
    if re.search(r"damaged|partial|recover|unrecover", s2):
        return "damage_status_not_stage"
    return "unknown"

def read_dirty_csv(path):
    last_error = None
    for enc in ["utf-8-sig", "utf-8", "latin1"]:
        try:
            return pd.read_csv(path, dtype=str, encoding=enc, engine="python", on_bad_lines="skip")
        except Exception as e:
            last_error = e
    raise last_error

# ------------------------------------------------------------
# Load cleaned file if present; otherwise create it from raw file
# ------------------------------------------------------------
if os.path.exists(CLEAN_UNIQUE_CSV):
    points_df = pd.read_csv(CLEAN_UNIQUE_CSV)
else:
    raw = read_dirty_csv(RAW_CSV)
    raw.columns = [str(c).strip() for c in raw.columns]

    df = raw.copy()

    df["year"] = df["Year"].map(parse_year)
    df["month"] = df["Month"].map(parse_month)
    df["region"] = df["Region"].map(clean_text)
    df["province"] = df["Province"].map(clean_text)
    df["municipality"] = df["Municipality"].map(clean_text)
    df["barangay"] = df["Barangay"].map(clean_text)

    df["latitude"] = df["Latitude"].map(lambda x: parse_coord(x, "lat"))
    df["longitude"] = df["Longitude"].map(lambda x: parse_coord(x, "lon"))

    df["crop_raw"] = df["Crop"].map(clean_text)
    df["variety_raw"] = df["Variety"].map(clean_text)
    df["growth_stage_raw"] = df["Growth stage"].map(clean_text)
    df["pest_raw"] = df["Pest"].map(clean_text)

    df["crop_std"] = np.where(
        df["crop_raw"].str.lower().str.contains("rice|palay", regex=True, na=False),
        "rice",
        "other"
    )

    df["pest_group"] = df["pest_raw"].map(standardize_rice_pest)
    df["stage_group"] = df["growth_stage_raw"].map(standardize_stage)

    rice = df[
        (df["crop_std"] == "rice") &
        (df["pest_group"].isin(TOP10_RICE_GROUPS)) &
        (df["latitude"].between(-90, 90)) &
        (df["longitude"].between(-180, 180)) &
        (df["year"].notna()) &
        (df["month"].notna())
    ].copy()

    rice["year"] = rice["year"].astype(int)
    rice["month"] = rice["month"].astype(int)

    keep_cols = [
        "year", "month",
        "region", "province", "municipality", "barangay",
        "latitude", "longitude",
        "crop_std", "crop_raw",
        "variety_raw",
        "growth_stage_raw", "stage_group",
        "pest_raw", "pest_group"
    ]

    rice_clean = rice[keep_cols].copy()

    points_df = (
        rice_clean
        .groupby(["year", "month", "latitude", "longitude", "crop_std", "pest_group"], dropna=False)
        .agg(
            n_reports=("pest_group", "size"),
            province=("province", "first"),
            municipality=("municipality", "first"),
            barangay=("barangay", "first"),
            stage_group=("stage_group", lambda x: ";".join(sorted(set([v for v in x if v])))),
            growth_stage_raw=("growth_stage_raw", lambda x: ";".join(sorted(set([v for v in x if v])))),
            pest_raw=("pest_raw", lambda x: ";".join(sorted(set([v for v in x if v])))),
            variety_raw=("variety_raw", lambda x: ";".join(sorted(set([v for v in x if v]))[:5]))
        )
        .reset_index()
    )

    points_df.to_csv(CLEAN_UNIQUE_CSV, index=False)

points_df["year"] = points_df["year"].astype(int)
points_df["month"] = points_df["month"].astype(int)
points_df["latitude"] = points_df["latitude"].astype(float)
points_df["longitude"] = points_df["longitude"].astype(float)

print("points_df:", points_df.shape)
display(points_df.head())
display(points_df.groupby(["pest_group", "month"]).size().reset_index(name="n").sort_values("n", ascending=False).head(30))
print("Saved/loaded:", CLEAN_UNIQUE_CSV)

In [ ]:
import ee
import geemap
import ipywidgets as widgets
from IPython.display import display, clear_output

ee.Authenticate()
ee.Initialize(project="sensesaka2024")

# ============================================================
# SETTINGS
# ============================================================

# Year-agnostic/recent combined mode
YEAR_MIN = 2021
YEAR_MAX = 2024
# Set YEAR_MAX = 2025 later only if GEE climate predictors work for 2025.

TRAINING_YEARS = [
    y for y in sorted(points_df["year"].unique().tolist())
    if YEAR_MIN <= int(y) <= YEAR_MAX
]

YEAR_LABEL = f"{YEAR_MIN}_{YEAR_MAX}"

QUARTERS = {
    "Q1": [1, 2, 3],
    "Q2": [4, 5, 6],
    "Q3": [7, 8, 9],
    "Q4": [10, 11, 12]
}

SCALE = 1000

# Fast/visual settings
N_RANDOM_BACKGROUND_PER_YEAR = 600
MAX_SURVEY_BACKGROUND_PER_YEAR = 600
MAX_PRESENCE_POINTS_PER_PEST = 3000
MIN_PRESENCE_POINTS = 30

# Use both random cropland + same-period other-pest survey background.
BACKGROUND_MODE = "mixed_same_period"

# For first visualization, keep this False to avoid queueing 40 Drive exports.
# Turn True once maps look good.
EXPORT_TO_DRIVE = False

OUT_DIR = GEE_OUTPUTS_DIR

VIS_PROB = {
    "min": 0,
    "max": 1,
    "palette": ["white", "yellow", "orange", "red", "darkred"]
}

print("Training years:", TRAINING_YEARS)

def safe_name(x):
    return (
        str(x).lower()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("-", "_")
        .replace("(", "")
        .replace(")", "")
    )

def clean_props(row):
    props = {}
    for k, v in row.items():
        if k in ["latitude", "longitude"]:
            continue
        if pd.isna(v):
            props[k] = None
        elif isinstance(v, (np.integer, int)):
            props[k] = int(v)
        elif isinstance(v, (np.floating, float)):
            props[k] = float(v)
        else:
            props[k] = str(v)
    return props

def pandas_points_to_ee_fc(df, lat_col="latitude", lon_col="longitude"):
    features = []
    for _, row in df.iterrows():
        lat = row[lat_col]
        lon = row[lon_col]
        if pd.isna(lat) or pd.isna(lon):
            continue
        geom = ee.Geometry.Point([float(lon), float(lat)])
        features.append(ee.Feature(geom, clean_props(row)))
    return ee.FeatureCollection(features)

# Philippines ROI
roi = (
    ee.FeatureCollection("FAO/GAUL/2015/level0")
    .filter(ee.Filter.eq("ADM0_NAME", "Philippines"))
    .geometry()
)

# Cropland mask: ESA WorldCover class 40
worldcover = ee.ImageCollection("ESA/WorldCover/v200").first().select("Map")
cropland = worldcover.eq(40).selfMask().clip(roi)

def monthly_predictors(year, month):
    start = ee.Date.fromYMD(int(year), int(month), 1)
    end = start.advance(1, "month")

    chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").select("precipitation")

    rain_30d = chirps.filterDate(start.advance(-30, "day"), start).sum().rename("rain_30d")
    rain_60d = chirps.filterDate(start.advance(-60, "day"), start).sum().rename("rain_60d")
    rain_month = chirps.filterDate(start, end).sum().rename("rain_month")

    modis = (
        ee.ImageCollection("MODIS/061/MOD13Q1")
        .filterDate(start.advance(-16, "day"), end.advance(16, "day"))
        .select(["NDVI", "EVI"])
        .median()
        .multiply(0.0001)
        .rename(["ndvi", "evi"])
    )

    tc = (
        ee.ImageCollection("IDAHO_EPSCOR/TERRACLIMATE")
        .filterDate(start, end)
        .mean()
    )

    tc_vars = tc.select(
        ["pr", "tmmx", "tmmn", "soil", "vpd", "aet", "pet"],
        ["tc_pr", "tc_tmmx", "tc_tmmn", "tc_soil", "tc_vpd", "tc_aet", "tc_pet"]
    )

    elev = ee.Image("USGS/SRTMGL1_003").select("elevation").rename("elevation")
    slope = ee.Terrain.slope(elev).rename("slope")

    month_sin = ee.Image.constant(np.sin(int(month) * 2 * np.pi / 12)).rename("month_sin").toFloat()
    month_cos = ee.Image.constant(np.cos(int(month) * 2 * np.pi / 12)).rename("month_cos").toFloat()

    img = (
        rain_30d
        .addBands(rain_60d)
        .addBands(rain_month)
        .addBands(modis)
        .addBands(tc_vars)
        .addBands(elev)
        .addBands(slope)
        .addBands(month_sin)
        .addBands(month_cos)
        .clip(roi)
        .toFloat()
    )

    return img

def period_predictors(year, months):
    imgs = []

    for m in months:
        img = monthly_predictors(year, m)

        # Remove month_sin/cos before averaging to avoid GEE homogeneous collection error.
        keep_bands = img.bandNames().removeAll(["month_sin", "month_cos"])
        imgs.append(img.select(keep_bands).toFloat())

    base = ee.ImageCollection.fromImages(imgs).mean().toFloat()

    mid_month = float(np.mean(months))
    month_sin = ee.Image.constant(np.sin(mid_month * 2 * np.pi / 12)).rename("month_sin").toFloat()
    month_cos = ee.Image.constant(np.cos(mid_month * 2 * np.pi / 12)).rename("month_cos").toFloat()

    return base.addBands(month_sin).addBands(month_cos).toFloat()

def combined_output_predictors(years, months):
    imgs = [period_predictors(y, months) for y in years]
    return ee.ImageCollection.fromImages(imgs).mean().toFloat()

def filter_year_month_df(df, years, months):
    return df[
        df["year"].astype(int).isin([int(y) for y in years]) &
        df["month"].astype(int).isin([int(m) for m in months])
    ].copy()

def make_presence_df(points_df, pest, years, months):
    df_period = filter_year_month_df(points_df, years, months)
    presence_df = df_period[df_period["pest_group"] == pest].copy()

    presence_df = presence_df[
        presence_df["latitude"].between(-90, 90) &
        presence_df["longitude"].between(-180, 180)
    ].copy()

    presence_df = presence_df.drop_duplicates(
        subset=["year", "month", "latitude", "longitude", "pest_group"]
    )

    if len(presence_df) > MAX_PRESENCE_POINTS_PER_PEST:
        presence_df = presence_df.sample(MAX_PRESENCE_POINTS_PER_PEST, random_state=42)

    return presence_df

def make_survey_background_df(points_df, pest, year, months):
    df_period = filter_year_month_df(points_df, [year], months)
    bg_df = df_period[df_period["pest_group"] != pest].copy()

    bg_df = bg_df[
        bg_df["latitude"].between(-90, 90) &
        bg_df["longitude"].between(-180, 180)
    ].copy()

    bg_df = bg_df.drop_duplicates(
        subset=["year", "month", "latitude", "longitude"]
    )

    if len(bg_df) > MAX_SURVEY_BACKGROUND_PER_YEAR:
        bg_df = bg_df.sample(MAX_SURVEY_BACKGROUND_PER_YEAR, random_state=42)

    return bg_df

def make_random_background_fc(n_pixels):
    return (
        cropland.sample(
            region=roi,
            scale=SCALE,
            numPixels=n_pixels,
            seed=42,
            geometries=True,
            tileScale=4
        )
        .map(lambda f: f.set({
            "presence": 0,
            "background_type": "random_cropland"
        }))
    )

def make_year_background_fc(points_df, pest, year, months):
    random_fc = make_random_background_fc(N_RANDOM_BACKGROUND_PER_YEAR)

    if BACKGROUND_MODE == "random_only":
        return random_fc, N_RANDOM_BACKGROUND_PER_YEAR, 0

    survey_bg_df = make_survey_background_df(points_df, pest, year, months)
    n_survey = len(survey_bg_df)

    if n_survey > 0:
        survey_fc = pandas_points_to_ee_fc(survey_bg_df)
        survey_fc = survey_fc.map(lambda f: f.set({
            "presence": 0,
            "background_type": "same_period_other_pest"
        }))
    else:
        survey_fc = ee.FeatureCollection([])

    return random_fc.merge(survey_fc), N_RANDOM_BACKGROUND_PER_YEAR, n_survey

def train_maxent(training_samples, band_list):
    training_samples = training_samples.filter(
        ee.Filter.notNull(list(band_list) + ["presence"])
    )

    return ee.Classifier.amnhMaxent(
        outputFormat="cloglog",
        writeClampGrid=False,
        seed=42
    ).train(
        features=training_samples,
        classProperty="presence",
        inputProperties=list(band_list)
    )

In [ ]:
# ============================================================
# RUN ALL QUARTERS x TOP PESTS
# Creates one year-agnostic map per pest per quarter.
# ============================================================

prob_maps = {}
presence_layers = {}
background_layers = {}
model_summary_rows = []
export_tasks = []

for q_label, months in QUARTERS.items():

    print("\n################################################")
    print("QUARTER:", q_label, "months:", months)
    print("################################################")

    predictor_img_output = combined_output_predictors(TRAINING_YEARS, months)
    BANDS = predictor_img_output.bandNames().getInfo()

    print("Predictor bands:", BANDS)

    for pest in TOP10_RICE_GROUPS:

        print("\n==============================")
        print("Running:", pest, q_label)
        print("==============================")

        presence_df = make_presence_df(
            points_df=points_df,
            pest=pest,
            years=TRAINING_YEARS,
            months=months
        )

        n_presence = len(presence_df)
        print("Presence records:", n_presence)

        if n_presence < MIN_PRESENCE_POINTS:
            print("Skipping; too few presence records.")
            model_summary_rows.append({
                "pest_group": pest,
                "year_label": YEAR_LABEL,
                "quarter": q_label,
                "months": ",".join(map(str, months)),
                "n_presence": n_presence,
                "status": "skipped_too_few_presence"
            })
            continue

        sample_fcs = []
        presence_fcs_for_map = []
        background_fcs_for_map = []

        total_random_bg = 0
        total_survey_bg = 0

        for yy in TRAINING_YEARS:

            yy_presence_df = presence_df[
                presence_df["year"].astype(int) == int(yy)
            ].copy()

            if len(yy_presence_df) == 0:
                continue

            yy_predictor_img = period_predictors(yy, months)

            yy_presence_fc = pandas_points_to_ee_fc(yy_presence_df)
            yy_presence_fc = yy_presence_fc.map(lambda f: f.set({
                "presence": 1,
                "background_type": "presence_target_pest",
                "sample_year": int(yy)
            }))

            yy_background_fc, yy_n_random_bg, yy_n_survey_bg = make_year_background_fc(
                points_df=points_df,
                pest=pest,
                year=yy,
                months=months
            )

            yy_background_fc = yy_background_fc.map(lambda f: f.set({
                "sample_year": int(yy)
            }))

            total_random_bg += yy_n_random_bg
            total_survey_bg += yy_n_survey_bg

            yy_training_points = (
                yy_presence_fc
                .merge(yy_background_fc)
                .randomColumn("split", seed=42)
            )

            yy_samples = yy_predictor_img.sampleRegions(
                collection=yy_training_points,
                properties=["presence", "split", "background_type", "sample_year"],
                scale=SCALE,
                geometries=True,
                tileScale=4
            )

            yy_samples = yy_samples.filter(
                ee.Filter.notNull(BANDS + ["presence", "split"])
            )

            sample_fcs.append(yy_samples)
            presence_fcs_for_map.append(yy_presence_fc)
            background_fcs_for_map.append(yy_background_fc)

        if len(sample_fcs) == 0:
            print("Skipping; no valid yearly samples.")
            model_summary_rows.append({
                "pest_group": pest,
                "year_label": YEAR_LABEL,
                "quarter": q_label,
                "n_presence": n_presence,
                "status": "skipped_no_valid_yearly_samples"
            })
            continue

        samples = ee.FeatureCollection(sample_fcs[0])
        for sfc in sample_fcs[1:]:
            samples = samples.merge(sfc)

        training_samples = samples.filter(ee.Filter.lt("split", 0.7))
        validation_samples = samples.filter(ee.Filter.gte("split", 0.7))

        n_train = training_samples.size().getInfo()
        n_val = validation_samples.size().getInfo()

        print("Random background total:", total_random_bg)
        print("Survey-effort background total:", total_survey_bg)
        print("Training samples:", n_train)
        print("Validation samples:", n_val)

        if n_train < 50 or n_val < 20:
            print("Skipping; too few samples after masking.")
            model_summary_rows.append({
                "pest_group": pest,
                "year_label": YEAR_LABEL,
                "quarter": q_label,
                "months": ",".join(map(str, months)),
                "n_presence": n_presence,
                "n_random_background": total_random_bg,
                "n_survey_background": total_survey_bg,
                "n_train": n_train,
                "n_validation": n_val,
                "status": "skipped_too_few_samples_after_masking"
            })
            continue

        try:
            classifier = train_maxent(training_samples, BANDS)

            probability = (
                predictor_img_output
                .select(BANDS)
                .classify(classifier)
                .rename("pest_occurrence_suitability")
                .updateMask(cropland)
                .clip(roi)
            )

            key = f"{pest} | {YEAR_LABEL}-{q_label}"
            prob_maps[key] = probability

            presence_fc_all = ee.FeatureCollection(presence_fcs_for_map[0])
            for pfc in presence_fcs_for_map[1:]:
                presence_fc_all = presence_fc_all.merge(pfc)

            background_fc_all = ee.FeatureCollection(background_fcs_for_map[0])
            for bfc in background_fcs_for_map[1:]:
                background_fc_all = background_fc_all.merge(bfc)

            presence_layers[key] = presence_fc_all
            background_layers[key] = background_fc_all

            print("Map stored:", key)

            model_summary_rows.append({
                "pest_group": pest,
                "year_label": YEAR_LABEL,
                "training_years": ",".join(map(str, TRAINING_YEARS)),
                "quarter": q_label,
                "months": ",".join(map(str, months)),
                "background_mode": BACKGROUND_MODE,
                "n_presence": n_presence,
                "n_random_background": total_random_bg,
                "n_survey_background": total_survey_bg,
                "n_train": n_train,
                "n_validation": n_val,
                "status": "ok"
            })

            if EXPORT_TO_DRIVE:
                export_name = f"pestprob_rice_{safe_name(pest)}_{YEAR_LABEL}_{q_label}"

                task = ee.batch.Export.image.toDrive(
                    image=probability,
                    description=export_name,
                    folder="PestRiskGEE",
                    fileNamePrefix=export_name,
                    region=roi,
                    scale=SCALE,
                    maxPixels=1e13
                )

                task.start()

                export_tasks.append({
                    "pest_group": pest,
                    "quarter": q_label,
                    "export_name": export_name,
                    "task": task
                })

                print("Export started:", export_name)

        except Exception as e:
            print("FAILED:", pest, q_label, e)

            model_summary_rows.append({
                "pest_group": pest,
                "year_label": YEAR_LABEL,
                "quarter": q_label,
                "months": ",".join(map(str, months)),
                "background_mode": BACKGROUND_MODE,
                "n_presence": n_presence,
                "n_random_background": total_random_bg,
                "n_survey_background": total_survey_bg,
                "status": "failed",
                "error": str(e)
            })

model_summary = pd.DataFrame(model_summary_rows)

summary_path = os.path.join(
    OUT_DIR,
    f"rice_top10_yearagnostic_allquarters_summary_{YEAR_LABEL}.csv"
)

model_summary.to_csv(summary_path, index=False)

print("\n================================================")
print("DONE")
print("Probability maps generated:", len(prob_maps))
print("Summary saved:", summary_path)
print("================================================")

display(model_summary)
print(list(prob_maps.keys()))

# ============================================================
# INTERACTIVE MAP
# ============================================================

if len(prob_maps) == 0:
    print("No maps generated. Check model_summary.")
    display(model_summary)
else:
    keys = list(prob_maps.keys())

    map_dropdown = widgets.Dropdown(
        options=keys,
        value=keys[0],
        description="Map:",
        layout=widgets.Layout(width="750px")
    )

    show_presence = widgets.Checkbox(value=True, description="Presence")
    show_background = widgets.Checkbox(value=False, description="Background")
    map_out = widgets.Output()

    def show_selected_map(change=None):
        selected = map_dropdown.value

        with map_out:
            clear_output(wait=True)

            m = geemap.Map()
            m.centerObject(roi, 6)

            m.addLayer(
                cropland,
                {"palette": ["#cccccc"]},
                "ESA WorldCover cropland mask",
                shown=False,
                opacity=0.35
            )

            m.addLayer(
                prob_maps[selected].unmask(0),
                VIS_PROB,
                f"Occurrence suitability: {selected}",
                shown=True,
                opacity=0.85
            )

            if show_presence.value and selected in presence_layers:
                m.addLayer(
                    presence_layers[selected],
                    {"color": "blue"},
                    f"Presence: {selected}",
                    shown=True
                )

            if show_background.value and selected in background_layers:
                m.addLayer(
                    background_layers[selected],
                    {"color": "gray"},
                    f"Background: {selected}",
                    shown=False
                )

            m.addLayerControl()
            display(m)

    map_dropdown.observe(show_selected_map, names="value")
    show_presence.observe(show_selected_map, names="value")
    show_background.observe(show_selected_map, names="value")

    display(widgets.HBox([map_dropdown, show_presence, show_background]))
    show_selected_map()
    display(map_out)

In [ ]:
# ============================================================
# EXPORT ALL GENERATED PEST MAPS TO GOOGLE DRIVE
# Meaningful file names:
# rice_maxent_occurrence_suitability_<pest>_<years>_<quarter>_ph_cropland_1km
# ============================================================

import ee
import os
import re
import pandas as pd

ee.Initialize(project="sensesaka2024")

EXPORT_FOLDER = "PestRiskGEE_Rice_YearAgnostic_Quarterly"
EXPORT_SCALE = SCALE  # usually 1000
EXPORT_CRS = "EPSG:4326"

def clean_export_text(x):
    x = str(x).lower().strip()
    x = re.sub(r"[^a-z0-9]+", "_", x)
    x = re.sub(r"_+", "_", x)
    x = x.strip("_")
    return x

def parse_map_key(key):
    # expected key: "Brown planthopper | 2021_2024-Q3"
    if "|" in key:
        pest_part, period_part = key.split("|", 1)
        pest = pest_part.strip()
        period = period_part.strip()
    else:
        pest = key.strip()
        period = f"{YEAR_LABEL}_unknown_period"

    return pest, period

export_tasks = []
export_rows = []

for key, img in prob_maps.items():

    pest, period = parse_map_key(key)

    pest_clean = clean_export_text(pest)
    period_clean = clean_export_text(period)

    export_name = (
        f"rice_maxent_occurrence_suitability_"
        f"{pest_clean}_{period_clean}_ph_cropland_1km"
    )

    # Keep task description shorter/safe
    task_description = export_name[:95]

    export_img = (
        img
        .rename("occurrence_suitability")
        .toFloat()
        .clip(roi)
    )

    task = ee.batch.Export.image.toDrive(
        image=export_img,
        description=task_description,
        folder=EXPORT_FOLDER,
        fileNamePrefix=export_name,
        region=roi,
        scale=EXPORT_SCALE,
        crs=EXPORT_CRS,
        maxPixels=1e13,
        fileFormat="GeoTIFF"
    )

    task.start()

    export_tasks.append({
        "map_key": key,
        "pest_group": pest,
        "period": period,
        "export_name": export_name,
        "task": task
    })

    export_rows.append({
        "map_key": key,
        "pest_group": pest,
        "period": period,
        "drive_folder": EXPORT_FOLDER,
        "file_name_prefix": export_name,
        "scale_m": EXPORT_SCALE,
        "crs": EXPORT_CRS,
        "band_name": "occurrence_suitability"
    })

    print("Export started:", export_name)

export_manifest = pd.DataFrame(export_rows)

manifest_path = os.path.join(
    OUT_DIR,
    f"export_manifest_rice_yearagnostic_quarterly_{YEAR_LABEL}.csv"
)

export_manifest.to_csv(manifest_path, index=False)

print("\nDONE.")
print("Number of export tasks:", len(export_tasks))
print("Drive folder:", EXPORT_FOLDER)
print("Manifest saved:", manifest_path)

display(export_manifest.head(20))

# Adding crop growth stage

Cells 0–3 above are the original environment-only workflow. They produce one 1 km suitability map per pest per quarter, exported to Drive by cell 3.

**Where the stage comes from.** The AI-CropGrowth pipeline writes a rice growth-stage map every run. Each period `YYYYMM` gets one 30 m HLS GeoTIFF per province: `rice_<PROVINCE>_<YYYYMM>.tiff`, with the dominant stage in the month. The classes are:

| Code | Stage | Weight used |
|---|---|---|
| -1 | outside the rice mask | nodata |
| 0 | no rice cycle detected | off-season |
| 1 | pre-planting / transplanting | establishment |
| 2 | vegetative | vegetative |
| 3 | reproductive | reproductive |
| 4 | ripening / maturity | ripening |
| 5 | harvested / post-harvest | off-season |

**The cells:**

- **Cell 5:** stage of each pest report, from the growth-stage text and `DAT`/`DAS` numbers, and the **pest × stage weight table** (`rice_top10_pest_stage_weights.csv`). The table blends what the reports show with an expert starting table.
- **Cell 10 (operational; run after every growth-stage run):** for each new period,

  `risk(30 m pixel) = suitability(1 km cell, quarter of the period) × weight[pest, stage of the pixel]`

  It writes, per period:
  - a 1 km risk GeoTIFF per pest (mean over the standing crop and over all rice area)
  - 1 km stage fractions
  - a province table of rice hectares per risk class
  - optional 30 m risk maps

  The code is in `pestrisk/stage.py` of this repository.
- **Cells 6–9 (optional, historical):** the growth-stage product only exists from 2026, while the pest models train on 2021–2024. These cells estimate stage from MODIS flooding for past years, to test whether stage helps as a model input. Once AI-CropGrowth is run for past periods, `pestrisk.stage.sample_stage_at_points` gives the product's stage at each report and replaces them.

Run cells 0, 1, 3 (with cell 2) and 5 once per model update. After that, cell 10 alone is enough each time new stage products appear.


In [ ]:
# ============================================================
# STAGE 1: GROWTH STAGE OF EACH REPORT + PEST x STAGE SUSCEPTIBILITY
# Needs points_df from cell 0. Works on the cached CSV too: the
# raw stage text is kept in growth_stage_raw (";"-joined per point).
# ============================================================

import re
import numpy as np
import pandas as pd

# Stage classes, ordered by crop development. Code 0 = off-season / harvested.
STAGES = ["establishment", "vegetative", "reproductive", "ripening"]
STAGE_CODE = {s: i + 1 for i, s in enumerate(STAGES)}
STAGE_CODE["off_season"] = 0

# Days after field establishment (transplanting or direct seeding) that end each stage.
# Typical for 110-120 day inbreds; tune with the dat_reported values below.
#   establishment: 0-20 | vegetative: 21-50 | reproductive (PI to flowering): 51-80
#   ripening (milk to maturity): 81-115 | > 115: off-season / harvested
STAGE_DAT_BREAKS = [20, 50, 80, 115]
STAGE_DAT_MID = {1: 10, 2: 35, 3: 65, 4: 98}   # used for stage-conditional maps
DAT_CAP = 150                                   # values above this are clamped

def stage_from_dat(dat):
    if dat is None or pd.isna(dat) or dat < 0:
        return "unknown"
    for code, upper in enumerate(STAGE_DAT_BREAKS, start=1):
        if dat <= upper:
            return STAGES[code - 1]
    return "off_season"

# "30 DAT", "45-50 DAS", "60 days after transplanting", "21 days after sowing"
_DAT_RE = re.compile(
    r"(\d{1,3})(?:\s*(?:-|to)\s*(\d{1,3}))?\s*"
    r"(?:dat|das|dae|days?\s+after\s+(?:trans\w*|sow\w*|seed\w*|plant\w*|emerg\w*|establish\w*))\b",
    flags=re.I,
)

def parse_reported_dat(text):
    """Days after establishment written in the stage text, or NaN."""
    m = _DAT_RE.search(str(text))
    if not m:
        return np.nan
    a = float(m.group(1))
    b = float(m.group(2)) if m.group(2) else a
    return (a + b) / 2.0

# Keyword rules, most advanced stage first (IRRI: milk/dough = ripening, not reproductive).
_STAGE_RULES = [
    ("ripening",      r"milk|dough|ripen|matur|harvest|grain\s*fill|hard\s*grain"),
    ("reproductive",  r"panicle\s*init|\bpi\b|boot|head|flower|anthesis|reproductive|panicle\s*exser"),
    ("vegetative",    r"vegetative|tiller|elongation|jointing"),
    ("establishment", r"nursery|seedbed|seedling|germin|emerg|sow|re\s*plant|transplant|establish"),
]

def parse_stage(text):
    """Stage of one raw stage string. A DAT/DAS number wins over keywords
    (the original standardize_stage put '45 days after transplanting' in establishment)."""
    s = clean_text(text).lower()
    if not s:
        return "unknown"
    dat = parse_reported_dat(s)
    if not np.isnan(dat):
        return stage_from_dat(dat)
    s2 = re.sub(r"[^a-z0-9]+", " ", s)
    for stage, pat in _STAGE_RULES:
        if re.search(pat, s2):
            return stage
    return "unknown"   # includes damage-status text such as "partially damaged"

def point_stage(growth_stage_raw):
    """One stage per aggregated point: most frequent known stage, ties go to the later stage."""
    parts = [p for p in str(growth_stage_raw).split(";") if p.strip()] if pd.notna(growth_stage_raw) else []
    stages = [parse_stage(p) for p in parts]
    stages = [s for s in stages if s in STAGES]
    dats = [parse_reported_dat(p) for p in parts]
    dats = [d for d in dats if not np.isnan(d)]
    dat = float(np.median(dats)) if dats else np.nan
    if not stages:
        return "unknown", dat
    counts = pd.Series(stages).value_counts()
    top = counts[counts == counts.max()].index
    return max(top, key=lambda s: STAGE_CODE[s]), dat

_ps = points_df["growth_stage_raw"].map(point_stage)
points_df["stage_main"] = _ps.map(lambda t: t[0])
points_df["dat_reported"] = _ps.map(lambda t: t[1])

print("Points by reported stage:")
display(points_df["stage_main"].value_counts().rename("n_points").to_frame())
print("Points with a DAT/DAS number:", int(points_df["dat_reported"].notna().sum()))

STAGE_POINTS_CSV = os.path.join(GEE_INPUTS_DIR, "rice_top10_pest_points_unique_with_stage.csv")
points_df.to_csv(STAGE_POINTS_CSV, index=False)
print("Saved:", STAGE_POINTS_CSV)

# ------------------------------------------------------------
# Pest x stage susceptibility weights (0-1)
# ------------------------------------------------------------
# Expert starting table. These are approximate, from general rice IPM knowledge.
# REVIEW WITH ENTOMOLOGISTS / PATHOLOGISTS BEFORE OPERATIONAL USE.
STAGE_PRIOR = pd.DataFrame(
    {
        #                        est  veg  rep  rip
        "Brown planthopper":     [0.3, 0.7, 1.0, 0.8],  # builds up from tillering; hopperburn near flowering-ripening
        "Stem borer":            [0.3, 1.0, 1.0, 0.3],  # deadheart (vegetative), whitehead (booting-heading)
        "Rodents":               [0.8, 0.5, 1.0, 1.0],  # seeds/seedlings, then booting to ripening
        "Bacterial leaf blight": [0.6, 0.8, 1.0, 0.6],  # kresek at seedling; most severe heading
        "Leaffolder":            [0.2, 0.8, 1.0, 0.3],  # flag-leaf damage most costly
        "Rice blast":            [0.8, 1.0, 1.0, 0.4],  # leaf blast seedling-tillering, neck blast at heading
        "Rice black bug":        [0.3, 0.8, 1.0, 0.6],  # tillering to maturity; bugburn around booting
        "Rice bug complex":      [0.0, 0.1, 0.6, 1.0],  # feeds on grain from flowering to soft dough
        "Bacterial leaf streak": [0.5, 1.0, 0.8, 0.4],
        "Green leafhopper":      [0.8, 1.0, 0.5, 0.2],  # tungro vector, young crop most at risk
    },
    index=STAGES,
).T

# Weight for pixels that are off-season / harvested (dat > last break).
W_OFFSEASON = {p: 0.0 for p in TOP10_RICE_GROUPS}
W_OFFSEASON["Rodents"] = 0.3   # rodents stay around fallow fields

# How many staged reports it takes before the data outweighs the prior (equal weight at this count).
PRIOR_STRENGTH = 50

known = points_df[points_df["stage_main"].isin(STAGES)]
stage_counts = (
    pd.crosstab(known["pest_group"], known["stage_main"], values=known["n_reports"], aggfunc="sum")
    .reindex(index=TOP10_RICE_GROUPS, columns=STAGES)
    .fillna(0)
)

# Relative risk P(stage | pest) / P(stage | all pests): corrects for surveys
# simply happening more often at some stages.
p_stage_given_pest = stage_counts.div(stage_counts.sum(axis=1).replace(0, np.nan), axis=0)
p_stage_all = stage_counts.sum(axis=0) / max(stage_counts.values.sum(), 1)
rel_risk = p_stage_given_pest.div(p_stage_all.replace(0, np.nan), axis=1)
empirical = rel_risk.div(rel_risk.max(axis=1), axis=0).fillna(0)

n_staged = stage_counts.sum(axis=1)
data_share = n_staged / (n_staged + PRIOR_STRENGTH)
STAGE_WEIGHTS = (
    empirical.mul(data_share, axis=0) + STAGE_PRIOR.mul(1 - data_share, axis=0)
)
STAGE_WEIGHTS = STAGE_WEIGHTS.div(STAGE_WEIGHTS.max(axis=1), axis=0).round(3)

print("\nStaged report counts (n_reports):")
display(stage_counts.astype(int))
print("Relative risk P(stage|pest)/P(stage|all):")
display(rel_risk.round(2))
print("Final stage weights (data share per pest in last column):")
display(STAGE_WEIGHTS.assign(data_share=data_share.round(2)))

weights_path = os.path.join(GEE_OUTPUTS_DIR, "rice_top10_pest_stage_weights.csv")
STAGE_WEIGHTS.assign(n_staged=n_staged, data_share=data_share.round(3)).to_csv(weights_path)
print("Saved:", weights_path)


In [ ]:
# ============================================================
# STAGE 2 (OPTIONAL, HISTORICAL): RICE STAGE FROM MODIS (days since field establishment)
# Only for years before the AI-CropGrowth product exists. Operational maps use cell 10.
#
# Paddies are flooded at puddling/transplanting/direct seeding. On MODIS
# 8-day surface reflectance (MOD09A1, 500 m) a flooded field with little
# canopy shows LSWI + 0.05 >= EVI or NDVI (Xiao et al. 2005, 2006).
# For a reference date (the 15th of the month):
#   1. last_flood = latest flooded composite in the past STAGE_LOOKBACK_DAYS
#   2. sos        = first flooded composite of that flooding episode
#                   (within FLOOD_EPISODE_MAX_DAYS before last_flood)
#   3. dat        = reference date - sos      (days since establishment)
#   4. stage      = dat cut at STAGE_DAT_BREAKS; dat > last break = off-season
# Pixels with no flooding in the window are masked. That also drops
# most non-paddy cropland, so the signal doubles as a rice mask.
# ============================================================

STAGE_LOOKBACK_DAYS = 150     # longer than one season (~115 days + land prep)
FLOOD_EPISODE_MAX_DAYS = 40   # flooding with low canopy lasts ~2-6 weeks
FLOOD_EVI_MAX = 0.35          # flooded-field test only below this EVI
EPOCH_DAY_MS = 86400000

STAGE_VIS = {
    "min": 0, "max": 4,
    # off-season, establishment, vegetative, reproductive, ripening
    "palette": ["#bdbdbd", "#4fc3f7", "#2e7d32", "#fbc02d", "#e65100"],
}

def _mod09_flood_days(img):
    sr = img.select(["sur_refl_b01", "sur_refl_b02", "sur_refl_b03", "sur_refl_b06"]).multiply(0.0001)
    red, nir, blue, swir = sr.select(0), sr.select(1), sr.select(2), sr.select(3)

    qa = img.select("StateQA")
    clear = (
        qa.bitwiseAnd(3).eq(0)                     # cloud state = clear
        .And(qa.bitwiseAnd(1 << 2).eq(0))          # no cloud shadow
        .And(sr.reduce(ee.Reducer.min()).gt(-0.01))  # no fill values
    )

    ndvi = nir.subtract(red).divide(nir.add(red))
    evi = nir.subtract(red).multiply(2.5).divide(
        nir.add(red.multiply(6)).subtract(blue.multiply(7.5)).add(1)
    )
    lswi = nir.subtract(swir).divide(nir.add(swir))

    flooded = (
        lswi.add(0.05).gte(evi).Or(lswi.add(0.05).gte(ndvi))
        .And(evi.lt(FLOOD_EVI_MAX))
    )

    t_days = ee.Image.constant(img.date().millis().divide(EPOCH_DAY_MS)).rename("t_days").toFloat()
    return t_days.updateMask(clear.And(flooded)).set("system:time_start", img.get("system:time_start"))

def rice_stage_image(year, month):
    """Bands: dat (days since establishment, clamped to DAT_CAP) and stage_code (0-4).
    Masked where no flooding was seen in the look-back window or outside cropland."""
    t_ref = ee.Date.fromYMD(int(year), int(month), 15)
    t_ref_days = t_ref.millis().divide(EPOCH_DAY_MS)

    flood_days = (
        ee.ImageCollection("MODIS/061/MOD09A1")
        .filterDate(t_ref.advance(-STAGE_LOOKBACK_DAYS, "day"), t_ref.advance(1, "day"))
        .map(_mod09_flood_days)
    )

    last_flood = flood_days.max()
    episode_start = last_flood.subtract(FLOOD_EPISODE_MAX_DAYS)
    sos = flood_days.map(lambda im: im.updateMask(im.gte(episode_start))).min()

    dat = (
        ee.Image.constant(t_ref_days).subtract(sos)
        .max(0).min(DAT_CAP)
        .rename("dat").toFloat()
    )

    b1, b2, b3, b4 = STAGE_DAT_BREAKS
    stage = (
        ee.Image(0)
        .where(dat.lte(b1), 1)
        .where(dat.gt(b1).And(dat.lte(b2)), 2)
        .where(dat.gt(b2).And(dat.lte(b3)), 3)
        .where(dat.gt(b3).And(dat.lte(b4)), 4)
        .updateMask(dat.mask())
        .rename("stage_code")
        .toByte()
    )

    return dat.addBands(stage).updateMask(cropland).clip(roi)

# ------------------------------------------------------------
# Check the satellite stage against the reported stage
# ------------------------------------------------------------
STAGE_CHECK_MAX_POINTS = 3000

def sample_stage_at_points(df):
    """Samples rice_stage_image for each point's own year and month."""
    fcs = []
    for (yy, mm), g in df.groupby(["year", "month"]):
        if len(g) == 0:
            continue
        fc = pandas_points_to_ee_fc(g[["year", "month", "latitude", "longitude", "stage_main", "dat_reported"]])
        img = rice_stage_image(yy, mm)
        fcs.append(img.sampleRegions(collection=fc, scale=SCALE, geometries=False, tileScale=4))
    if not fcs:
        return pd.DataFrame()
    merged = ee.FeatureCollection(fcs).flatten()
    cols = ["stage_main", "dat_reported", "dat", "stage_code"]
    rows = merged.reduceColumns(ee.Reducer.toList(len(cols)), cols).get("list").getInfo()
    # reduceColumns drops rows with a null, so dat_reported must not be NaN (see -1 below)
    return pd.DataFrame(rows, columns=cols)

check_df = points_df[
    points_df["stage_main"].isin(STAGES)
    & points_df["year"].isin(TRAINING_YEARS)
].drop_duplicates(subset=["year", "month", "latitude", "longitude"]).copy()
check_df["dat_reported"] = check_df["dat_reported"].fillna(-1)   # -1 = no number reported

if len(check_df) > STAGE_CHECK_MAX_POINTS:
    check_df = check_df.sample(STAGE_CHECK_MAX_POINTS, random_state=42)

print("Points with a reported stage to check:", len(check_df))
stage_check = sample_stage_at_points(check_df)

if len(stage_check):
    code_to_stage = {v: k for k, v in STAGE_CODE.items()}
    stage_check["stage_rs"] = stage_check["stage_code"].astype(int).map(code_to_stage)

    print("Points with a satellite stage:", len(stage_check),
          f"({len(stage_check) / len(check_df):.0%}; the rest had no flooding signal)")

    confusion = pd.crosstab(
        stage_check["stage_main"], stage_check["stage_rs"],
        rownames=["reported"], colnames=["satellite"]
    ).reindex(index=STAGES, columns=["off_season"] + STAGES, fill_value=0)
    display(confusion)

    agree = (stage_check["stage_main"] == stage_check["stage_rs"]).mean()
    rep_code = stage_check["stage_main"].map(STAGE_CODE)
    within_one = ((rep_code - stage_check["stage_code"]).abs() <= 1).mean()
    print(f"Same stage: {agree:.0%}   Within one stage: {within_one:.0%}")

    with_num = stage_check[stage_check["dat_reported"] >= 0]
    if len(with_num) >= 10:
        diff = with_num["dat"] - with_num["dat_reported"]
        print(f"DAT, satellite minus reported (n={len(with_num)}): "
              f"median {diff.median():.0f} days, MAE {diff.abs().mean():.0f} days, "
              f"r = {np.corrcoef(with_num['dat'], with_num['dat_reported'])[0, 1]:.2f}")
        print("A consistent median offset means FLOOD_EPISODE_MAX_DAYS or STAGE_DAT_BREAKS need tuning.")

    stage_check.to_csv(os.path.join(GEE_OUTPUTS_DIR, "rice_stage_satellite_vs_reported.csv"), index=False)
else:
    print("No points could be checked.")

# Quick look at one month
_preview = rice_stage_image(TRAINING_YEARS[-1], 8)
m = geemap.Map()
m.centerObject(roi, 6)
m.addLayer(_preview.select("stage_code"), STAGE_VIS, f"Rice stage {TRAINING_YEARS[-1]}-08")
m.addLayer(_preview.select("dat"), {"min": 0, "max": 120, "palette": ["#e3f2fd", "#1b5e20"]},
           f"Days since establishment {TRAINING_YEARS[-1]}-08", shown=False)
m.addLayerControl()
print("Stage colours: grey off-season, blue establishment, green vegetative, yellow reproductive, orange ripening")
display(m)


In [ ]:
# ============================================================
# STAGE 3: TRAIN ENV-ONLY AND ENV+STAGE MAXENT ON THE SAME SAMPLES
#
# Differences from cell 2:
#  - Points are sampled month by month, so each point gets the stage
#    (dat) of its own month. Climate predictors stay quarterly, as before.
#  - Survey background drops places where the target pest was also
#    reported in the same month (they were labelled both 1 and 0).
#  - Random background uses a different seed per year and month.
#  - Two models per pest x quarter: env-only (BANDS) and env+stage
#    (BANDS + dat). Both get a validation AUC.
#  - Output: one map per pest x quarter x stage, with dat fixed at the
#    middle of that stage ("risk if the crop is at this stage").
# ============================================================

# "drop": points with no flooding signal are left out (default; keeps
#         training on paddy pixels and the same samples for both models).
# "offseason": they get dat = DAT_CAP instead.
DAT_MISSING = "drop"

def dat_for_model(year, month):
    dat = rice_stage_image(year, month).select("dat")
    fill = DAT_CAP if DAT_MISSING == "offseason" else -1
    return dat.unmask(fill).clip(roi).rename("dat").toFloat()

def random_background_fc(n_pixels, seed):
    return (
        cropland.sample(region=roi, scale=SCALE, numPixels=n_pixels,
                        seed=int(seed), geometries=True, tileScale=4)
        .map(lambda f: f.set({"presence": 0, "background_type": "random_cropland"}))
    )

def survey_background_df(points_df, pest, year, month, presence_df, cap):
    bg = filter_year_month_df(points_df, [year], [month])
    bg = bg[bg["pest_group"] != pest].drop_duplicates(subset=["year", "month", "latitude", "longitude"])
    # drop locations where the target pest was reported in the same month
    pres_keys = set(zip(presence_df["year"], presence_df["month"],
                        presence_df["latitude"].round(5), presence_df["longitude"].round(5)))
    keys = list(zip(bg["year"], bg["month"], bg["latitude"].round(5), bg["longitude"].round(5)))
    bg = bg[[k not in pres_keys for k in keys]]
    if len(bg) > cap:
        bg = bg.sample(cap, random_state=42)
    return bg

def auc_score(labels, scores):
    """ROC AUC via the Mann-Whitney rank formula (ties get average ranks)."""
    labels = np.asarray(labels, dtype=int)
    scores = np.asarray(scores, dtype=float)
    n_pos, n_neg = labels.sum(), (1 - labels).sum()
    if n_pos == 0 or n_neg == 0:
        return np.nan
    ranks = pd.Series(scores).rank(method="average").values
    return float((ranks[labels == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))

def validation_auc(classifier, fc):
    """AUC of the classifier on a sampled FeatureCollection with a 'presence' property."""
    try:
        scored = fc.classify(classifier)
        names = ee.Feature(scored.first()).propertyNames().getInfo()
        prop = next((p for p in ["probability", "classification"] if p in names), None)
        if prop is None:
            return np.nan
        rows = scored.reduceColumns(ee.Reducer.toList(2), ["presence", prop]).get("list").getInfo()
        arr = np.array(rows, dtype=float)
        return auc_score(arr[:, 0], arr[:, 1]) if len(arr) else np.nan
    except Exception as e:
        print("AUC failed:", e)
        return np.nan

STAGE_LABEL = {1: "establishment", 2: "vegetative", 3: "reproductive", 4: "ripening"}

stage_maps = {}          # "pest | years-Q | stage" -> image (env + stage model, dat fixed)
env_maps = {}            # "pest | years-Q"         -> image (env-only model)
classifiers = {}         # (pest, quarter) -> {"env": clf, "stage": clf, "bands_env": [...], "bands_stage": [...]}
stage_summary_rows = []

for q_label, months in QUARTERS.items():
    print("\n################ QUARTER", q_label, months, "################")

    predictor_img_output = combined_output_predictors(TRAINING_YEARS, months)
    BANDS = predictor_img_output.bandNames().getInfo()
    BANDS_STAGE = BANDS + ["dat"]

    for pest in TOP10_RICE_GROUPS:
        print("\n--", pest, q_label)

        presence_df = make_presence_df(points_df, pest, TRAINING_YEARS, months)
        n_presence = len(presence_df)
        row = {"pest_group": pest, "quarter": q_label, "months": ",".join(map(str, months)),
               "year_label": YEAR_LABEL, "n_presence": n_presence, "dat_missing": DAT_MISSING}

        if n_presence < MIN_PRESENCE_POINTS:
            print("Skipping; too few presence records:", n_presence)
            stage_summary_rows.append({**row, "status": "skipped_too_few_presence"})
            continue

        sample_fcs = []
        n_rand_per_month = max(1, N_RANDOM_BACKGROUND_PER_YEAR // len(months))
        survey_cap_per_month = max(1, MAX_SURVEY_BACKGROUND_PER_YEAR // len(months))

        for yy in TRAINING_YEARS:
            period_img = period_predictors(yy, months)
            for mm in months:
                pres_m = presence_df[(presence_df["year"] == yy) & (presence_df["month"] == mm)]
                if len(pres_m) == 0:
                    continue

                pres_fc = pandas_points_to_ee_fc(pres_m).map(
                    lambda f: f.set({"presence": 1, "background_type": "presence_target_pest"}))

                bg_fc = random_background_fc(n_rand_per_month, seed=yy * 100 + mm)
                if BACKGROUND_MODE != "random_only":
                    sbg = survey_background_df(points_df, pest, yy, mm, presence_df, survey_cap_per_month)
                    if len(sbg):
                        bg_fc = bg_fc.merge(pandas_points_to_ee_fc(sbg).map(
                            lambda f: f.set({"presence": 0, "background_type": "same_period_other_pest"})))

                pts = (
                    pres_fc.merge(bg_fc)
                    .map(lambda f: f.set({"sample_year": int(yy), "sample_month": int(mm)}))
                    .randomColumn("split", seed=42)
                )

                img = period_img.addBands(dat_for_model(yy, mm))
                s = img.sampleRegions(
                    collection=pts,
                    properties=["presence", "split", "background_type", "sample_year", "sample_month"],
                    scale=SCALE, geometries=True, tileScale=4,
                )
                sample_fcs.append(s.filter(ee.Filter.notNull(BANDS_STAGE + ["presence", "split"])))

        if not sample_fcs:
            stage_summary_rows.append({**row, "status": "skipped_no_valid_samples"})
            continue

        samples_all = ee.FeatureCollection(sample_fcs).flatten()
        samples = samples_all.filter(ee.Filter.gte("dat", 0))   # no-op in "offseason" mode

        train = samples.filter(ee.Filter.lt("split", 0.7))
        val = samples.filter(ee.Filter.gte("split", 0.7))

        counts = ee.Dictionary({
            "all": samples_all.size(),
            "all_presence": samples_all.filter(ee.Filter.eq("presence", 1)).size(),
            "kept_presence": samples.filter(ee.Filter.eq("presence", 1)).size(),
            "train": train.size(),
            "val": val.size(),
        }).getInfo()

        row.update({
            "n_samples_before_stage_mask": counts["all"],
            "n_presence_sampled": counts["all_presence"],
            "n_presence_with_stage": counts["kept_presence"],
            "n_train": counts["train"],
            "n_validation": counts["val"],
        })
        print("Presence with a stage:", counts["kept_presence"], "of", counts["all_presence"],
              "| train", counts["train"], "| val", counts["val"])

        if counts["train"] < 50 or counts["val"] < 20 or counts["kept_presence"] < MIN_PRESENCE_POINTS:
            print("Skipping; too few samples with a stage.")
            stage_summary_rows.append({**row, "status": "skipped_too_few_samples_with_stage"})
            continue

        try:
            clf_env = train_maxent(train, BANDS)
            clf_stage = train_maxent(train, BANDS_STAGE)

            auc_env = validation_auc(clf_env, val)
            auc_stage = validation_auc(clf_stage, val)
            print(f"Validation AUC  env-only {auc_env:.3f}  env+stage {auc_stage:.3f}")

            classifiers[(pest, q_label)] = {
                "env": clf_env, "stage": clf_stage,
                "bands_env": BANDS, "bands_stage": BANDS_STAGE,
            }

            env_key = f"{pest} | {YEAR_LABEL}-{q_label}"
            env_maps[env_key] = (
                predictor_img_output.select(BANDS).classify(clf_env)
                .rename("occurrence_suitability").updateMask(cropland).clip(roi)
            )

            for code, label in STAGE_LABEL.items():
                img = predictor_img_output.addBands(
                    ee.Image.constant(STAGE_DAT_MID[code]).rename("dat").toFloat()
                )
                stage_maps[f"{env_key} | {label}"] = (
                    img.select(BANDS_STAGE).classify(clf_stage)
                    .rename("occurrence_suitability").updateMask(cropland).clip(roi)
                )

            stage_summary_rows.append({
                **row, "auc_env": auc_env, "auc_env_stage": auc_stage,
                "auc_gain": auc_stage - auc_env, "status": "ok",
            })

        except Exception as e:
            print("FAILED:", e)
            stage_summary_rows.append({**row, "status": "failed", "error": str(e)})

stage_summary = pd.DataFrame(stage_summary_rows)
stage_summary_path = os.path.join(GEE_OUTPUTS_DIR, f"rice_top10_stage_model_summary_{YEAR_LABEL}.csv")
stage_summary.to_csv(stage_summary_path, index=False)

print("\nDONE. Stage-conditional maps:", len(stage_maps), "| env-only maps:", len(env_maps))
print("Summary saved:", stage_summary_path)
display(stage_summary)

if "auc_gain" in stage_summary:
    print("AUC gain from adding stage, by pest (mean over quarters):")
    display(stage_summary.groupby("pest_group")[["auc_env", "auc_env_stage", "auc_gain"]].mean().round(3)
            .sort_values("auc_gain", ascending=False))
    print("Note: the split is random, so nearby points fall in both sets and AUCs are optimistic. "
          "Compare the two models with each other, not with 1.0.")


In [ ]:
# ============================================================
# STAGE 4: RISK MAPS FOR A GIVEN MONTH (uses the real stage map)
#
# A. env+stage model:  climate of that month + satellite dat
# B. weighted:         env-only suitability x STAGE_WEIGHTS[pest, stage]
#                      (off-season pixels get W_OFFSEASON[pest])
# Pixels with no flooding signal (no rice season detected) are masked.
# ============================================================

NOWCAST_YEAR = 2024
NOWCAST_MONTH = 8

def quarter_of(month):
    return next(q for q, ms in QUARTERS.items() if int(month) in ms)

def month_env_predictors(year, month):
    """One month's predictors, with month_sin/cos set to the quarter's middle month
    as in training (the models never saw any other value)."""
    q_months = QUARTERS[quarter_of(month)]
    base = monthly_predictors(year, month)
    base = base.select(base.bandNames().removeAll(["month_sin", "month_cos"]))
    mid = float(np.mean(q_months))
    return (
        base
        .addBands(ee.Image.constant(np.sin(mid * 2 * np.pi / 12)).rename("month_sin").toFloat())
        .addBands(ee.Image.constant(np.cos(mid * 2 * np.pi / 12)).rename("month_cos").toFloat())
    )

def nowcast_maps(pest, year, month):
    q = quarter_of(month)
    if (pest, q) not in classifiers:
        return None
    c = classifiers[(pest, q)]

    stage_img = rice_stage_image(year, month)
    has_stage = stage_img.select("dat").mask()
    env = month_env_predictors(year, month)

    risk_stage_model = (
        env.addBands(stage_img.select("dat"))
        .select(c["bands_stage"]).classify(c["stage"])
        .rename("risk_stage_model")
    )

    suit_env = env.select(c["bands_env"]).classify(c["env"]).rename("suitability_env")
    w = STAGE_WEIGHTS.loc[pest]
    weight = stage_img.select("stage_code").remap(
        [0, 1, 2, 3, 4],
        [float(W_OFFSEASON[pest])] + [float(w[s]) for s in STAGES],
    ).rename("stage_weight")
    risk_weighted = suit_env.multiply(weight).rename("risk_stage_weighted")

    return (
        risk_stage_model.addBands(risk_weighted).addBands(suit_env).addBands(weight)
        .addBands(stage_img.select("stage_code"))
        .updateMask(has_stage).updateMask(cropland).clip(roi)
        .toFloat()
    )

nowcasts = {}
for pest in TOP10_RICE_GROUPS:
    img = nowcast_maps(pest, NOWCAST_YEAR, NOWCAST_MONTH)
    if img is not None:
        nowcasts[f"{pest} | {NOWCAST_YEAR}-{NOWCAST_MONTH:02d}"] = img
print("Monthly risk maps:", len(nowcasts))

# ------------------------------------------------------------
# Viewer: stage-conditional maps (cell 7) and monthly maps (this cell)
# ------------------------------------------------------------
view_options = (
    [("Stage-conditional | " + k, ("cond", k)) for k in stage_maps]
    + [("Monthly, env+stage model | " + k, ("now_model", k)) for k in nowcasts]
    + [("Monthly, stage-weighted | " + k, ("now_weighted", k)) for k in nowcasts]
    + [("Env-only | " + k, ("env", k)) for k in env_maps]
)

if not view_options:
    print("No maps. Run cell 7 first and check stage_summary.")
else:
    stage_dropdown = widgets.Dropdown(options=view_options, value=view_options[0][1],
                                      description="Map:", layout=widgets.Layout(width="850px"))
    show_stage = widgets.Checkbox(value=False, description="Stage map")
    stage_out = widgets.Output()

    def show_stage_view(change=None):
        kind, key = stage_dropdown.value
        if kind == "cond":
            img, label = stage_maps[key], f"Suitability if crop at stage: {key}"
        elif kind == "now_model":
            img, label = nowcasts[key].select("risk_stage_model"), f"Risk (env+stage model): {key}"
        elif kind == "now_weighted":
            img, label = nowcasts[key].select("risk_stage_weighted"), f"Risk (stage-weighted): {key}"
        else:
            img, label = env_maps[key], f"Env-only suitability: {key}"

        with stage_out:
            clear_output(wait=True)
            m = geemap.Map()
            m.centerObject(roi, 6)
            if show_stage.value:
                m.addLayer(rice_stage_image(NOWCAST_YEAR, NOWCAST_MONTH).select("stage_code"),
                           STAGE_VIS, f"Rice stage {NOWCAST_YEAR}-{NOWCAST_MONTH:02d}", opacity=0.7)
            m.addLayer(img.unmask(0), VIS_PROB, label, opacity=0.85)
            m.addLayerControl()
            display(m)

    stage_dropdown.observe(show_stage_view, names="value")
    show_stage.observe(show_stage_view, names="value")
    display(widgets.HBox([stage_dropdown, show_stage]))
    show_stage_view()
    display(stage_out)


In [ ]:
# ============================================================
# STAGE 5: EXPORT STAGE MAPS TO GOOGLE DRIVE
# Stage-conditional: 10 pests x 4 quarters x 4 stages = up to 160 tasks.
# Turn the switches on for what you need.
# ============================================================

EXPORT_STAGE_CONDITIONAL = True
EXPORT_MONTHLY = True
EXPORT_STAGE_MAP = True

STAGE_EXPORT_FOLDER = "PestRiskGEE_Rice_Stage"
EXPORT_SCALE = SCALE
EXPORT_CRS = "EPSG:4326"

def clean_export_text(x):
    x = re.sub(r"[^a-z0-9]+", "_", str(x).lower().strip())
    return re.sub(r"_+", "_", x).strip("_")

stage_export_rows = []

def start_export(img, name):
    task = ee.batch.Export.image.toDrive(
        image=img, description=name[:95], folder=STAGE_EXPORT_FOLDER,
        fileNamePrefix=name, region=roi, scale=EXPORT_SCALE, crs=EXPORT_CRS,
        maxPixels=1e13, fileFormat="GeoTIFF",
    )
    task.start()
    stage_export_rows.append({"file_name_prefix": name, "drive_folder": STAGE_EXPORT_FOLDER,
                              "bands": ",".join(img.bandNames().getInfo())})
    print("Export started:", name)

if EXPORT_STAGE_CONDITIONAL:
    for key, img in stage_maps.items():
        pest, period, stage = [p.strip() for p in key.split("|")]
        name = (f"rice_maxent_stage_suitability_{clean_export_text(pest)}_"
                f"{clean_export_text(period)}_{stage}_ph_cropland_1km")
        start_export(img.toFloat(), name)

if EXPORT_MONTHLY:
    for key, img in nowcasts.items():
        pest, ym = [p.strip() for p in key.split("|")]
        name = f"rice_stage_risk_{clean_export_text(pest)}_{ym.replace('-', '_')}_ph_cropland_1km"
        start_export(img, name)

if EXPORT_STAGE_MAP:
    name = f"rice_stage_modis_{NOWCAST_YEAR}_{NOWCAST_MONTH:02d}_ph_cropland"
    start_export(rice_stage_image(NOWCAST_YEAR, NOWCAST_MONTH).toFloat(), name)

stage_manifest = pd.DataFrame(stage_export_rows)
stage_manifest_path = os.path.join(GEE_OUTPUTS_DIR, f"export_manifest_rice_stage_{YEAR_LABEL}.csv")
stage_manifest.to_csv(stage_manifest_path, index=False)
print("\nTasks:", len(stage_manifest), "| Manifest:", stage_manifest_path)


In [ ]:
#@title 10. Operational: pest risk from the AI-CropGrowth growth-stage product
# ============================================================
# Run this after each growth-stage run. It processes every period
# (rice_<PROVINCE>_<YYYYMM>.tiff) that has no output yet, and redoes a
# period when its province files changed (e.g. a failed province was added).
#
#   risk(30 m pixel) = suitability(1 km, quarter of the period) x weight[pest, stage]
#
# Inputs:
#   STAGE_DIR    growth-stage products. The folder is shared from another
#                account: in Drive, right-click "auto" > Organize > Add shortcut,
#                and put the shortcut in My Drive.
#   SUIT_DIR     quarterly suitability GeoTIFFs exported by cell 3
#   WEIGHTS_CSV  pest x stage weights written by cell 5
#   REPO_DIR     copy of the ai-pestriskmapping repository (pestrisk/stage.py)
# ============================================================

import os, sys

STAGE_DIR = os.path.join(BASE_DRIVE, "auto")  #@param {type:"string"}
SUIT_DIR = os.path.join(BASE_DRIVE, "PestRiskGEE_Rice_YearAgnostic_Quarterly")  #@param {type:"string"}
WEIGHTS_CSV = os.path.join(GEE_OUTPUTS_DIR, "rice_top10_pest_stage_weights.csv")  #@param {type:"string"}
RISK_OUT_DIR = os.path.join(PEST_DIR, "pest_risk_stage")  #@param {type:"string"}
REPO_DIR = os.path.join(BASE_DRIVE, "ai-pestriskmapping")  #@param {type:"string"}
WRITE_30M = False  #@param {type:"boolean"}
ONLY_PERIODS = ""  #@param {type:"string"}
# ONLY_PERIODS: comma-separated YYYYMM to force a rerun, e.g. "202610". Empty = pending periods only.

%pip install -q rasterio

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
from pestrisk import stage as pest_stage

for label, path in [("STAGE_DIR", STAGE_DIR), ("SUIT_DIR", SUIT_DIR), ("WEIGHTS_CSV", WEIGHTS_CSV),
                    ("REPO_DIR", REPO_DIR)]:
    print(f"{label:12s} {'ok     ' if os.path.exists(path) else 'MISSING'} {path}")

# Off-season weight (class 0 no rice cycle, class 5 harvested), from cell 5 when it was run.
OFFSEASON = dict(W_OFFSEASON) if "W_OFFSEASON" in globals() else {"Rodents": 0.3}

products = pest_stage.list_stage_products(STAGE_DIR)
print("\nStage products:", len(products), "files,", products["period"].nunique(), "period(s):",
      ", ".join(sorted(products["period"].unique())))
print(pest_stage.read_product_info(products.iloc[0]["path"]))

periods = [p.strip() for p in ONLY_PERIODS.split(",") if p.strip()] or None
results = pest_stage.run_pending(
    STAGE_DIR, SUIT_DIR, WEIGHTS_CSV, RISK_OUT_DIR,
    offseason=OFFSEASON, periods=periods, write_30m=WRITE_30M,
)

# ------------------------------------------------------------
# Summary and a quick look at the latest period
# ------------------------------------------------------------
import matplotlib.pyplot as plt
import rasterio

latest = sorted(os.listdir(RISK_OUT_DIR))[-1] if os.path.isdir(RISK_OUT_DIR) else None
if latest:
    table = pd.read_csv(os.path.join(RISK_OUT_DIR, latest, f"rice_pest_risk_by_province_{latest}.csv"))
    print(f"\nPeriod {latest}: standing-crop hectares at high or very high risk")
    table["ha_high_plus"] = table["ha_high"] + table["ha_very_high"]
    display(table.pivot_table(index="province", columns="pest_group", values="ha_high_plus", aggfunc="sum")
            .round(0).sort_values(by=table["pest_group"].iloc[0], ascending=False).head(25))
    print("Rice hectares with no suitability value (outside the ESA cropland mask used in cell 2):",
          round(table.drop_duplicates("province")["ha_no_suitability"].sum()))

    pests_done = sorted(table["pest_group"].unique())
    fig, axes = plt.subplots(2, 5, figsize=(22, 10), constrained_layout=True)
    for ax, pest in zip(axes.ravel(), pests_done):
        with rasterio.open(os.path.join(RISK_OUT_DIR, latest,
                                        f"rice_pest_risk_{pest_stage.slug(pest)}_{latest}_1km.tif")) as src:
            r = src.read(1)
            b = src.bounds
        im = ax.imshow(r, vmin=0, vmax=1, cmap="YlOrRd", extent=[b.left, b.right, b.bottom, b.top])
        ax.set_title(pest, fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
    for ax in axes.ravel()[len(pests_done):]:
        ax.axis("off")
    fig.colorbar(im, ax=axes, shrink=0.6, label="Risk, standing crop (suitability x stage weight)")
    fig.suptitle(f"Rice pest risk {latest}")
    plt.show()
